# Module Output Artifacts Inspection

This notebook loads the final parquet artifacts from each module pipeline (the ones that get uploaded to HuggingFace) and displays their heads.

## Unified Schema

Each module produces three standardized parquet files:

1. **annotations.parquet**: `rsid, module, gene, phenotype, category`
2. **studies.parquet**: `rsid, module, pmid, population, p_value, conclusion, study_design`
3. **weights.parquet**: `rsid, genotype, module, weight, state, priority, conclusion, curator, method`
   - The uploaded weights file is joined with Ensembl data (includes chrom, start, end, ref, alts, clinvar, etc.)

In [2]:
from pathlib import Path
import polars as pl
from IPython.display import display

# Base output directory
MODULES_OUTPUT_DIR = Path("../data/output/modules")

# Available modules with their expected files
MODULES = [
    "longevitymap",
    "lipidmetabolism",
    "vo2max",
    "superhuman",
    "coronary",
    "drugs",
]

## Module Weights (Head Display)

For each module, shows row counts for annotations/studies and displays the **weights** head (the main file uploaded to HuggingFace).

In [3]:
for module in MODULES:
    module_dir = MODULES_OUTPUT_DIR / module
    
    if not module_dir.exists():
        print(f"⚠️  {module}: directory not found")
        continue
    
    # Annotations & Studies - just show row counts
    counts = []
    for file_type in ["annotations", "studies"]:
        path = module_dir / f"{file_type}.parquet"
        if path.exists():
            rows = pl.scan_parquet(path).select(pl.len()).collect().item()
            counts.append(f"{file_type}: {rows:,}")
    
    # Weights - show head
    weights_path = module_dir / f"{module}_ensembl_joined.parquet"
    if not weights_path.exists():
        weights_path = module_dir / "weights.parquet"
    
    if weights_path.exists():
        lf = pl.scan_parquet(weights_path)
        rows = lf.select(pl.len()).collect().item()
        print(f"\n{module.upper()} — {', '.join(counts)}, weights: {rows:,}")
        display(lf.head(5).collect())


LONGEVITYMAP — annotations: 520, studies: 3,460, weights: 1,386


rsid,genotype,module,weight,state,priority,conclusion,curator,method,chrom,start,end,ref,alts,clinvar,pathogenic,benign,likely_pathogenic,likely_benign
str,list[str],str,f64,str,str,str,str,str,str,u32,u32,str,list[str],bool,bool,bool,bool,bool
"""rs922943""","[""A"", ""C""]","""longevitymap""",0.07,"""protective""","""0.14""","""281 SNPs were found to discrim…","""Olga Borysova""","""literature_review""","""3""",25367257,25367257,"""G""","[""A"", ""C""]",false,false,false,false,false
"""rs11086106""","[""T"", ""T""]","""longevitymap""",0.14,"""protective""","""0.14""","""281 SNPs were found to discrim…","""Olga Borysova""","""literature_review""","""19""",18350648,18350648,"""C""","[""T""]",false,false,false,false,false
"""rs3120819""","[""C"", ""T""]","""longevitymap""",0.24,"""protective""","""0.48""","""A total of 27 SNPs were identi…","""Olga Borysova""","""literature_review""","""1""",11340352,11340352,"""A""","[""C"", ""T""]",false,false,false,false,false
"""rs4964728""","[""A"", ""T""]","""longevitymap""",0.105,"""protective""","""0.21""","""NDUFS1, TXNRD1, SOD2 and UCP3 …","""Olga Borysova""","""literature_review""","""12""",104255955,104255955,"""G""","[""A"", ""C"", ""T""]",false,false,false,false,false
"""rs7301631""","[""A"", ""C""]","""longevitymap""",0.105,"""protective""","""0.21""","""NDUFS1, TXNRD1, SOD2 and UCP3 …","""Olga Borysova""","""literature_review""","""12""",104279484,104279484,"""T""","[""A"", ""C"", ""G""]",false,false,false,false,false



LIPIDMETABOLISM — annotations: 16, studies: 16, weights: 45


rsid,genotype,module,weight,state,priority,conclusion,curator,method,chrom,start,end,ref,alts,clinvar,pathogenic,benign,likely_pathogenic,likely_benign
str,list[str],str,f64,str,str,str,str,str,str,u32,u32,str,list[str],bool,bool,bool,bool,bool
"""rs6756629""","[""A"", ""G""]","""lipidmetabolism""",-0.2,"""alt""",null,"""G allele is a risk allele. You…","""Olga Borysova""","""literature_review""","""2""",43837951,43837951,"""G""","[""A"", ""C"", ""T""]",false,false,true,false,true
"""rs6544713""","[""C"", ""C""]","""lipidmetabolism""",0.0,"""alt""",null,"""CC genotype is NOT associated …","""Olga Borysova""","""literature_review""","""2""",43846742,43846742,"""T""","[""A"", ""C""]",false,false,true,false,false
"""rs12740374""","[""G"", ""T""]","""lipidmetabolism""",-0.4,"""alt""",null,"""GT genotype is associated with…","""Olga Borysova""","""literature_review""","""1""",109274968,109274968,"""G""","[""T""]",false,false,false,false,false
"""rs268""","[""A"", ""G""]","""lipidmetabolism""",-0.5,"""alt""",null,"""You are heterozygous carrier o…","""Olga Borysova""","""literature_review""","""8""",19956018,19956018,"""A""","[""G""]",false,false,true,false,true
"""rs6544713""","[""T"", ""T""]","""lipidmetabolism""",-0.8,"""ref""",null,"""The allele T of this SNP is as…","""Olga Borysova""","""literature_review""","""2""",43846742,43846742,"""T""","[""A"", ""C""]",false,false,true,false,false



VO2MAX — annotations: 13, studies: 13, weights: 39


...

rsid,genotype,module,weight,state,priority,conclusion,curator,method,chrom,start,end,ref,alts,clinvar,pathogenic,benign,likely_pathogenic,likely_benign
str,list[str],str,f64,str,str,str,str,str,str,u32,u32,str,list[str],bool,bool,bool,bool,bool
"""rs7412""","[""T"", ""T""]","""superhuman""",null,"""protective""",null,"""Low risk of Alzheimer disease …","""Olga Borysova""","""literature_review""","""19""",44908822,44908822,"""C""","[""T""]",false,true,true,false,true
"""rs5082""","[""C"", ""C""]","""superhuman""",null,"""protective""",null,"""Low coronary disease | Adverse…","""Olga Borysova""","""literature_review""","""1""",161223893,161223893,"""G""","[""A"", ""C"", ""T""]",false,true,false,false,false



CORONARY — annotations: 27, studies: 79, weights: 81


rsid,genotype,module,weight,state,priority,conclusion,curator,method,chrom,start,end,ref,alts,clinvar,pathogenic,benign,likely_pathogenic,likely_benign
str,list[str],str,f64,str,str,str,str,str,str,u32,u32,str,list[str],bool,bool,bool,bool,bool
"""rs11206510""","[""T"", ""T""]","""coronary""",0.0,"""alt""",null,"""PCSK9 overexpression increases…","""Olga Borysova""","""gwas_literature""","""1""",55030366,55030366,"""T""","[""A"", ""C"", ""G""]",false,false,false,false,false
"""rs3184504""","[""C"", ""C""]","""coronary""",0.0,"""ref""",null,"""SH2B adaptor protein 3 (SH2B3)…","""Olga Borysova""","""gwas_literature""","""12""",111446804,111446804,"""T""","[""A"", ""C"", ""G""]",false,false,true,false,false
"""rs3184504""","[""T"", ""T""]","""coronary""",-0.7,"""alt""",null,"""SH2B adaptor protein 3 (SH2B3)…","""Olga Borysova""","""gwas_literature""","""12""",111446804,111446804,"""T""","[""A"", ""C"", ""G""]",false,false,true,false,false
"""rs7250581""","[""A"", ""A""]","""coronary""",0.0,"""alt""",null,"""rs7250581 has been reported in…","""Olga Borysova""","""gwas_literature""","""19""",29573489,29573489,"""A""","[""G""]",false,false,false,false,false
"""rs8055236""","[""T"", ""T""]","""coronary""",0.0,"""ref""",null,"""CDH13 encodes T-cadherin, is …","""Olga Borysova""","""gwas_literature""","""16""",83178793,83178793,"""G""","[""A"", ""C"", ""T""]",false,false,false,false,false



DRUGS — annotations: 740, studies: 854, weights: 854


rsid,genotype,module,weight,state,priority,conclusion,curator,method
str,str,str,f64,str,str,str,str,str
"""rs6822844""","""??""","""drugs""",null,"""significant""",null,"""rituximab: Genotype GG is asso…","""PharmGKB""","""pharmacogenomics_db"""
"""rs2314339""","""??""","""drugs""",null,"""significant""",null,"""lithium: Allele T is associate…","""PharmGKB""","""pharmacogenomics_db"""
"""rs20455""","""??""","""drugs""",null,"""significant""",null,"""pravastatin: Genotypes AG + GG…","""PharmGKB""","""pharmacogenomics_db"""
"""rs683369""","""??""","""drugs""",null,"""significant""",null,"""imatinib: Genotypes CG + GG is…","""PharmGKB""","""pharmacogenomics_db"""
"""rs1883112""","""??""","""drugs""",null,"""significant""",null,"""idarubicin: Genotype AA is ass…","""PharmGKB""","""pharmacogenomics_db"""


## Summary Table

Overview of all module output files with row counts and sizes.

In [4]:
summary_rows = []

for module in MODULES:
    module_dir = MODULES_OUTPUT_DIR / module
    
    if not module_dir.exists():
        continue
    
    for parquet_file in sorted(module_dir.glob("*.parquet")):
        lf = pl.scan_parquet(parquet_file)
        row_count = lf.select(pl.len()).collect().item()
        col_count = len(lf.collect_schema())
        size_mb = parquet_file.stat().st_size / (1024 * 1024)
        
        summary_rows.append({
            "module": module,
            "file": parquet_file.name,
            "rows": row_count,
            "columns": col_count,
            "size_mb": round(size_mb, 3),
        })

summary_df = pl.DataFrame(summary_rows)
print("\n📊 Summary of All Module Output Files:")
display(summary_df)


📊 Summary of All Module Output Files:


module,file,rows,columns,size_mb
str,str,i64,i64,f64
"""longevitymap""","""annotations.parquet""",520,5,0.006
"""longevitymap""","""longevitymap_ensembl_joined.pa…",1386,19,0.028
"""longevitymap""","""studies.parquet""",3460,7,0.079
"""longevitymap""","""weights.parquet""",1386,9,0.02
"""lipidmetabolism""","""annotations.parquet""",16,5,0.002
…,…,…,…,…
"""coronary""","""studies.parquet""",79,7,0.014
"""coronary""","""weights.parquet""",81,9,0.01
"""drugs""","""annotations.parquet""",740,5,0.006


## Files That Get Uploaded to HuggingFace

The following files are uploaded to `just-dna-seq/annotators` repository:

| Module | HuggingFace Path | Local Source |
|--------|------------------|-------------|
| longevitymap | `data/longevitymap/annotations.parquet` | `annotations.parquet` |
| longevitymap | `data/longevitymap/studies.parquet` | `studies.parquet` |
| longevitymap | `data/longevitymap/weights.parquet` | `longevitymap_ensembl_joined.parquet` |
| ... | ... | ... |

In [5]:
# List what would be uploaded for each module
upload_manifest = []

for module in MODULES:
    module_dir = MODULES_OUTPUT_DIR / module
    
    if not module_dir.exists():
        continue
    
    annotations_path = module_dir / "annotations.parquet"
    studies_path = module_dir / "studies.parquet"
    ensembl_joined_path = module_dir / f"{module}_ensembl_joined.parquet"
    weights_path = module_dir / "weights.parquet"
    
    # Determine weights source
    weights_source = ensembl_joined_path if ensembl_joined_path.exists() else weights_path
    
    for local_path, hf_name in [
        (annotations_path, "annotations.parquet"),
        (studies_path, "studies.parquet"),
        (weights_source, "weights.parquet"),
    ]:
        if local_path.exists():
            size_mb = local_path.stat().st_size / (1024 * 1024)
            row_count = pl.scan_parquet(local_path).select(pl.len()).collect().item()
            
            upload_manifest.append({
                "module": module,
                "hf_path": f"data/{module}/{hf_name}",
                "local_file": local_path.name,
                "rows": row_count,
                "size_mb": round(size_mb, 3),
            })

upload_df = pl.DataFrame(upload_manifest)
print("\n🚀 Files Ready for HuggingFace Upload:")
display(upload_df)

total_size = upload_df["size_mb"].sum()
total_rows = upload_df["rows"].sum()
print(f"\n📦 Total: {len(upload_manifest)} files, {total_rows:,} rows, {total_size:.2f} MB")


🚀 Files Ready for HuggingFace Upload:


module,hf_path,local_file,rows,size_mb
str,str,str,i64,f64
"""longevitymap""","""data/longevitymap/annotations.…","""annotations.parquet""",520,0.006
"""longevitymap""","""data/longevitymap/studies.parq…","""studies.parquet""",3460,0.079
"""longevitymap""","""data/longevitymap/weights.parq…","""longevitymap_ensembl_joined.pa…",1386,0.028
"""lipidmetabolism""","""data/lipidmetabolism/annotatio…","""annotations.parquet""",16,0.002
"""lipidmetabolism""","""data/lipidmetabolism/studies.p…","""studies.parquet""",16,0.009
…,…,…,…,…
"""coronary""","""data/coronary/studies.parquet""","""studies.parquet""",79,0.014
"""coronary""","""data/coronary/weights.parquet""","""coronary_ensembl_joined.parque…",81,0.009
"""drugs""","""data/drugs/annotations.parquet""","""annotations.parquet""",740,0.006



📦 Total: 18 files, 9,842 rows, 0.24 MB
